# Enclave Inference — Gemma 3 — Researcher

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `test.enclave@gmail.com` | Trusted execution environment |
| **Model owner** | `test.model.owner@gmail.com` | Owns the Gemma 3 model (weights + inference engine) |
| **Benchmark owner** | `test.benchmark.owner@gmail.com` | Owns AI safety evaluation prompts |
| **Researcher** | _you_ | Submits inference job for bias/safety evaluation |

This notebook drives only the Researcher steps; the Model Owner and Benchmark Owner run their own notebooks in parallel.

## Setup

In [ ]:
!uv pip install -Uq "git+https://github.com/OpenMined/syft-client.git@main#subdirectory=packages/syft-enclave"

In [ ]:
import json, os, random, tempfile
from pathlib import Path

os.environ["PRE_SYNC"] = "false"

from syft_enclaves import login_do, login_ds

In [ ]:
ENCLAVE_EMAIL = "test.enclave@gmail.com"
MODEL_OWNER_EMAIL = "test.model.owner@gmail.com"
BENCHMARK_OWNER_EMAIL = "test.benchmark.owner@gmail.com"

# In real life the Researcher would learn this from the Model Owner's mock model card; hardcoded here so the notebook is self-contained.
MODEL_SIZE = "270m"
CKPT_SUBDIR = "gemma-3-270m-it"

print(f"  Enclave: {ENCLAVE_EMAIL}  |  Model owner: {MODEL_OWNER_EMAIL}  |  Benchmark owner: {BENCHMARK_OWNER_EMAIL}  |  Model size: {MODEL_SIZE}")

## Step 0 — Log in as Researcher

In [ ]:
researcher = login_ds()
print(f"  Researcher : {researcher.email}")

In [ ]:
# Optionally to clear state
researcher._manager.delete_syftbox()
researcher._manager.peer_manager.write_own_version()

## Step 1 — Peer with Model Owner, Benchmark Owner, and the Enclave

In [ ]:
researcher.add_peer(MODEL_OWNER_EMAIL)
researcher.add_peer(BENCHMARK_OWNER_EMAIL)
researcher.add_peer(ENCLAVE_EMAIL)

researcher.sync()
print("  Peer requests sent to Model Owner, Benchmark Owner, and Enclave")

### Step 1.1 — Wait until both Data Owners approve, then verify peers

Each DO needs to run their `approve_peer_request` cell. Re-run the cell below until both appear as approved peers.

In [ ]:
researcher.sync()
researcher.peers

## Step 2 — Browse the mock datasets

In [ ]:
researcher.sync()
researcher.datasets

## Step 3 — Job helpers

In [ ]:
def create_code_file(code: str) -> str:
    tmp = Path(tempfile.mkdtemp()) / f"job-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "main.py"
    p.write_text(code)
    return str(p)

## Step 4 — Define the inference job

The job loads the Model Owner's `gemma_inference.py` + weights from the `gemma3_model` dataset and the Benchmark Owner's prompts from `safety_prompts`, runs Gemma inference inside the enclave, and writes `outputs/safety_eval_results.json`. The `dependencies=GEMMA_DEPS` argument tells the enclave to install JAX/Flax/Orbax/SentencePiece at job runtime.

In [ ]:
JOB_NAME = "safety_eval_job"
JOB_CODE = f'''
import json
import os

import syft_client as sc

# Resolve Model owner's private model dataset directory
model_files = sc.resolve_dataset_files_path(
    "gemma3_model", owner_email="{MODEL_OWNER_EMAIL}"
)
weights_dir = str(model_files[0].parent)

# Import the inference module from the model owner's private dataset
gemma = sc.load_dataset_code(
    "gemma3_model.gemma_inference", owner_email="{MODEL_OWNER_EMAIL}"
)

# Load model, tokenizer, and params in one call
print(f"Loading Gemma 3 {MODEL_SIZE.upper()}-IT from {{weights_dir}}...")
model, tokenizer, params = gemma.setup("{MODEL_SIZE}", weights_dir)
print("Model loaded successfully")

# Load Benchmark owner's private prompts (one per line)
prompt_path = sc.resolve_dataset_file_path(
    "safety_prompts", owner_email="{BENCHMARK_OWNER_EMAIL}"
)
prompts = [line for line in open(prompt_path).read().splitlines() if line.strip()]
print(f"Loaded {{len(prompts)}} evaluation prompts")

# Run inference on each prompt
results = []
for i, prompt in enumerate(prompts):
    print(f"  [{{i+1}}/{{len(prompts)}}] {{prompt[:50]}}...")
    completion, stats = gemma.generate(
        model, params, tokenizer, prompt,
        max_new_tokens=100, temperature=0.8, top_k=40,
    )
    results.append({{
        "prompt": prompt,
        "completion": completion,
        "ttft": stats["ttft"],
        "decode_tps": stats["decode_tps"],
    }})

# Write outputs
os.makedirs("outputs", exist_ok=True)
with open("outputs/safety_eval_results.json", "w") as f:
    json.dump({{
        "model": "{CKPT_SUBDIR}",
        "total_prompts": len(results),
        "results": results,
    }}, f, indent=2)

print(f"\\nInference complete. {{len(results)}} prompts evaluated.")
'''

## Step 5 — Submit the job to the enclave

In [ ]:
GEMMA_DEPS = ["jax[cpu]", "flax", "orbax-checkpoint", "sentencepiece"]

researcher.submit_python_job(
    ENCLAVE_EMAIL,
    create_code_file(JOB_CODE),
    JOB_NAME,
    datasets={
        MODEL_OWNER_EMAIL: ["gemma3_model"],
        BENCHMARK_OWNER_EMAIL: ["safety_prompts"],
    },
    share_results_with_do=True,
    dependencies=GEMMA_DEPS,
)
researcher.sync()
print(f"  Job '{JOB_NAME}' submitted to the enclave ({ENCLAVE_EMAIL})")

## Step 6 — Wait for both DOs to approve and the enclave to run

Each DO must approve from their notebook; once both approve, the enclave runs the job and pushes results back. Re-sync until status is `"done"`.

In [ ]:
researcher.sync()
researcher_job = next(j for j in researcher.jobs if j.name == JOB_NAME)
print(f"  Researcher job status : {researcher_job.status}")
print(f"  Output files          : {researcher_job.output_paths}")

assert researcher_job.status == "done", researcher_job.status
assert len(researcher_job.output_paths) > 0

## Step 7 — View the results

In [ ]:
with open(researcher_job.output_paths[0]) as f:
    result = json.load(f)

print()
print(f"  Model                   : {result['model']}")
print(f"  Total prompts evaluated : {result['total_prompts']}")
print()
for r in result["results"]:
    completion = r["completion"]
    print(f"  prompt     : {r['prompt']}")
    print(f"  completion : {completion[:120]}..." if len(completion) > 120 else f"  completion : {completion}")
    print(f"  TTFT={r['ttft']:.2f}s  decode={r['decode_tps']:.1f} tok/s")
    print()